1) Create conda or venv environment on head node <br>
2) Activate environment <br>
3) Install required packages (scanpy) + jupyterlab <br>
4) Submit interactive job from head node: <br>
    ```bsub -G team361 -q teramem -n4 -M 400000 -R"select[mem>400000 ] rusage[mem=400000 ]" -Is /bin/bash``` <br>
5) Activate environment <br>
6) Run  <br>
    ```jupyter-lab --no-browser --ip=0.0.0.0 --port=8888 &``` <br>
7) Forward port on local machine: <br>
    ```ssh -o TCPKeepAlive=no -o ServerAliveInterval=15 -L 8882:b11-hm-01:8888 sb75@farm22-head1.internal.sanger.ac.uk``` <br>
8) Open browser and go to ```http://localhost:8882/```

In [ ]:
import scanpy as sc

In [ ]:
sc.__version__

In [ ]:
artifacts_folder_path = '../../artifacts'
silver_folder_path = '/lustre/scratch126/cellgen/team361/DATASETS/silver'

load_timestamp = '06032025_085433_744'
read_checkpoint = 'ep2'

In [ ]:
adata = sc.read_h5ad(f'{artifacts_folder_path}/hst_corpus_70m/{load_timestamp}/adata_{read_checkpoint}_1000-1007-1008_combined.h5ad')

In [ ]:
adata

In [ ]:
adata.obs.head(1)

In [ ]:
# Cell embedding
adata.obsm['cell_emb_layer_6']

In [ ]:
# Neighborhood embedding
adata.obsm['neighborhood_emb_layer_6']

In [ ]:
adata.obs[['dataset', 'batch', 'cell']] = [s.split('_') for s in adata.obs.index]
adata.obs['dataset_batch'] = adata.obs['dataset'].astype(str) + adata.obs['batch'].astype(str)
adata.obs['cell'] = adata.obs['cell'].astype(int)

In [ ]:
# Perform stratified subsampling based on batch
sample_frac = 0.001
random_state = 42
sampled_df = adata.obs.groupby(['dataset', 'batch'], group_keys=False).apply(
        lambda x: x.sample(frac=sample_frac, replace=False, random_state=random_state))
adata_subset = adata[sampled_df.index]
del(sampled_df)

In [ ]:
sc.__version__

In [ ]:
# Neighborhood embedding UMAP
sc.pp.neighbors(adata_subset,
                n_neighbors=15,
                use_rep='neighborhood_emb_layer_6',
                key_added='neighborhood')

sc.tl.umap(adata_subset,
           neighbors_key='neighborhood',
           key_added='neighborhood_umap')

label = 'tissue'
sc.pl.umap(adata_subset,
           color=label)

In [ ]:
# Cell embedding UMAP
sc.pp.neighbors(adata_subset,
                n_neighbors=15,
                use_rep='cell_emb_layer_6',
                key_added='cell')

sc.tl.umap(adata_subset,
           neighbors_key='cell',
           key_added='cell_umap')

label = 'tissue'
sc.pl.umap(adata_subset,
           color=label)

In [ ]:
# 1000: eczema Lloyd
# 1007: kidney Daniyal
# 1008: embryo Adam

# xhk1007-CV1-41b_1p
# xhk1007-CV2-41b_1p
# xhk1007-CV3-41b_1p
# xhk1007-CV4-41b_1p
# xhk1007-CV5-41b_1p
# xhk1007-CV6-41b_1p
# xhk1007-CV7-41b_1p
# xhk1007-CV8-41b_1p
# xhk1007-CV9-41b_1p


adata_raw = sc.read_h5ad(f'{silver_folder_path}/xhk1007-CV1-41b_1p/CV1-KID-0-FO-1.h5ad')

In [ ]:
# original_cell_id column

'sample_batch_map': {
    'CV1-KID-0-FO-1': 36,
    'CV1-KID-0-FT-1': 14,
    'CV1-KID-0-FT-2': 20,
    'CV1-KID-0-FT-3': 1,
    'CV2-KID-0-FO-1': 17,
    'CV2-KID-0-FO-2': 33,
    'CV2-KID-0-FT-1-s2': 4,
    'CV2-KID-0-FT-2': 2,
    'CV3-KID-0-FT-1-s3': 21,
    'CV3-KID-0-FT-2-s3': 28,
    'CV3-KID-0-FT-3-s3': 35,
    'CV3-KID-0-FT-4-s3': 8,
    'CV3-KID-0-FT-5': 16,
    'CV4-KID-0-FO-1': 37,
    'CV4-KID-0-FT-1-s3': 9,
    'CV4-KID-0-FT-2-s3': 23,
    'CV4-KID-0-FT-3-S3': 40,
    'CV4-KID-0-FT-4-S3': 10,
    'CV5-KID-0-FO-1': 31,
    'CV5-KID-0-FT-1-s3': 38,
    'CV5-KID-0-FT-2-s3': 5,
    'CV5-KID-0-FT-4-S3': 7,
    'CV6-KID-0-FO-1': 0,
    'CV6-KID-0-FT-1': 27,
    'CV6-KID-0-FT-2-s3': 3,
    'CV6-KID-0-FT-3-s3': 12,
    'CV6-KID-0-FT-4-S3': 19,
    'CV6-KID-0-FT-5-S3': 24,
    'CV7-KID-0-FO-1': 26,
    'CV7-KID-0-FT-1-s3': 29,
    'CV7-KID-0-FT-2-s3': 32,
    'CV7-KID-0-FT-3-S3': 15,
    'CV7-KID-0-FT-4-S3': 13,
    'CV8-KID-0-FO-1': 18,
    'CV8-KID-0-FT-1-s3': 11,
    'CV9-KID-0-FO-1': 6,
    'CV9-KID-0-FT-1-s3': 25,
    'CV9-KID-0-FT-2-S3': 34,
    'CV9-KID-0-FT-3-S3': 30,
    'CV9-KID-0-FT-4-S3': 39,
    'CV9-KID-0-FT-5-S4': 22}

In [ ]:
adata_raw.obs.head(1)

In [ ]:
adata_raw.X.toarray()